# 01 - Ingesta Bronze
Este notebook implementa la ingesta batch del dataset de operaciones BYMA hacia la capa bronze.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbfs_path = "dbfs:/Volumes/workspace/default/iol_data/operaciones_byma_2026.csv"

In [0]:
schema = StructType([
    StructField("fecha", StringType(), True),
    StructField("tipoTran", StringType(), True),
    StructField("id_cliente", StringType(), True),
    StructField("descripcion_titulo", StringType(), True),
    StructField("moneda", StringType(), True),
    StructField("simbolo_titulo", StringType(), True),
    StructField("cantidad", IntegerType(), True),
    StructField("precio", DecimalType(18, 4), True),
    StructField("id_transaccion", StringType(), True),
    StructField("origen", StringType(), True)
])

In [0]:
df_raw = spark.read \
    .option("header", True) \
    .schema(schema) \
    .csv(dbfs_path)

In [0]:
df = df_raw \
    .withColumn("fecha_ts_utc", to_timestamp("fecha")) \
    .withColumn("fecha_ts_local", from_utc_timestamp("fecha_ts_utc", "America/Argentina/Buenos_Aires")) \
    .withColumn("fecha_particion", to_date("fecha_ts_local")) \
    .withColumn("_ingested_at", current_timestamp())

In [0]:
#Duplicados
df_duplicates = df.groupBy("id_transaccion") \
    .count() \
    .filter(col("count") > 1) \
    .withColumn("fecha_deteccion", current_timestamp())

In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS bronze_byma")

In [0]:
df_duplicates.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_byma.calidad_duplicados")

In [0]:
#Procesar por día (batch diario)
fechas = [row["fecha_particion"] for row in df.select("fecha_particion").distinct()]
fechas = sorted(fechas)

In [0]:
from delta.tables import DeltaTable

table_name = "bronze_byma.operaciones_raw"

for fecha in fechas:
    
    df_batch_day = df.filter(col("fecha_particion") == fecha)
    
    if spark.catalog.tableExists(table_name):
        delta_table = DeltaTable.forName(spark, table_name)
        
        delta_table.alias("target").merge(
            df_batch_day.alias("source"),
            "target.id_transaccion = source.id_transaccion"
        ).whenNotMatchedInsertAll().execute()
        
    else:
        df_batch_day.write \
            .format("delta") \
            .partitionBy("fecha_particion") \
            .saveAsTable(table_name)

# Se utiliza delta para lograr idempotente (re-ejecutarlo no genera duplicados)

Validaciones

In [0]:
df.count()

In [0]:
%sql
SELECT COUNT(*) FROM bronze_byma.operaciones_raw

In [0]:
%sql
SELECT id_transaccion, COUNT(*) 
FROM bronze_byma.operaciones_raw
GROUP BY id_transaccion
HAVING COUNT(*) > 1

In [0]:
%sql
SHOW PARTITIONS bronze_byma.operaciones_raw

In [0]:
display(df.select("fecha", "fecha_ts_utc", "fecha_ts_local").limit(10))

In [0]:
%sql
SELECT MIN(precio), MAX(precio)
FROM bronze_byma.operaciones_raw

In [0]:
%sql
SELECT moneda, COUNT(*)
FROM bronze_byma.operaciones_raw
GROUP BY moneda

In [0]:
%sql
SELECT tipoTran, COUNT(*)
FROM bronze_byma.operaciones_raw
GROUP BY tipoTran

In [0]:
%sql
SELECT COUNT(*) 
FROM bronze_byma.operaciones_raw
WHERE id_transaccion IS NULL